In [1]:
import os
import shutil

# 1. Retreat to base and clean
os.chdir('/kaggle/working')
repo_path = '/kaggle/working/Patch-DM'
if os.path.exists(repo_path):
    print("🧹 Cleaning up old repository...")
    shutil.rmtree(repo_path)

# 2. Fresh Clone
!git clone https://github.com/mlpc-ucsd/Patch-DM.git
%cd Patch-DM

# 3. Dependencies
!pip install -q pytorch-lightning==1.9.5 lmdb==1.2.1 lpips pytorch-fid einops
!pip install -q git+https://github.com/openai/CLIP.git

print("\n✅ Environment clean and ready.")

Cloning into 'Patch-DM'...
remote: Enumerating objects: 93, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 93 (delta 21), reused 85 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (93/93), 1.06 MiB | 9.37 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/kaggle/working/Patch-DM
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 881.5/881.5 kB 15.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/829.5 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.6 MB/s eta 0:00:00

✅ Environment clean and ready.


In [2]:
import os
import re

# --- 1. Fix model/unet.py (Novelty Injection) ---
with open('model/unet.py', 'r') as f:
    unet_lines = f.readlines()

blender_class = [
    "\nclass GatedWeightBlender(nn.Module):\n",
    "    def __init__(self, channels):\n",
    "        super().__init__()\n",
    "        self.gate = nn.Sequential(\n",
    "            nn.Conv2d(channels, channels // 2, kernel_size=1),\n",
    "            nn.SiLU(),\n",
    "            nn.Conv2d(channels // 2, 1, kernel_size=1),\n",
    "            nn.Sigmoid()\n",
    "        )\n",
    "    def forward(self, x_orig, x_shifted):\n",
    "        alpha = self.gate(x_orig)\n",
    "        return alpha * x_orig + (1 - alpha) * x_shifted\n\n"
]
unet_lines.insert(15, "".join(blender_class))
unet_content = "".join(unet_lines)

# Inject initialization and forward logic
init_anchor = "if conf.resnet_use_zero_module:"
unet_content = unet_content.replace(init_anchor, f"self.blender = GatedWeightBlender(ch)\n        {init_anchor}", 1)

forward_anchor = "pred = self.out(h)"
blending_logic = "h_blended = self.blender(h, torch.roll(h, shifts=(2,2), dims=(2,3)))\n        pred = self.out(h_blended)"
unet_content = unet_content.replace(forward_anchor, blending_logic, 1)

with open('model/unet.py', 'w') as f:
    f.write(unet_content)

# --- 2. Fix experiment.py (Overfit Hack + Logger + Stability) ---
with open('experiment.py', 'r') as f:
    exp_content = f.read()

# Sub-select only 100 images for rapid convergence
subset_hack = """self.train_data = self.conf.make_dataset()
        from torch.utils.data import Subset
        if self.train_data is not None:
            self.train_data = Subset(self.train_data, range(100))"""
exp_content = exp_content.replace('self.train_data = self.conf.make_dataset()', subset_hack)

# Add Loss Logger (Prints every 1000 steps)
logger_code = """
class ConsoleLossLogger(pl.Callback):
    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        if trainer.global_step % 1000 == 0:
            loss = outputs['loss'].item()
            print(f"   [Step {trainer.global_step}] Loss: {loss:.4f} | Epoch: {trainer.current_epoch}")
"""
exp_content += "\n" + logger_code

# Fix Stability & Hooks
exp_content = exp_content.replace('from numpy.lib.function_base import flip', 'from numpy import flip')
exp_content = re.sub(
    r'def on_train_batch_end\s*\([^)]+\)\s*->\s*None:',
    r'def on_train_batch_end(self, outputs, batch, batch_idx: int) -> None:',
    exp_content
)

# 7 HOUR TIMEOUT FOR DEADLINE SAFETY
exp_content = exp_content.replace("desc='infer')", "desc='infer', disable=True)")
exp_content = exp_content.replace('trainer = pl.Trainer(', 'trainer = pl.Trainer(enable_progress_bar=False, max_time="00:07:00:00", ')
exp_content = exp_content.replace('callbacks = [LearningRateMonitor()]', 'callbacks = [LearningRateMonitor(), ConsoleLossLogger()]')

with open('experiment.py', 'w') as f:
    f.write(exp_content)

# --- 3. Fix train.py (Frequency) ---
with open('train.py', 'r') as f:
    train_content = f.read()
train_content = train_content.replace('conf.save_every_samples = 100_000', 'conf.save_every_samples = 10_000')
with open('train.py', 'w') as f:
    f.write(train_content)

print("🚀 Overfit constraint (100 images), Novelty, and 7-hour timer applied.")

🚀 Overfit constraint (100 images), Novelty, and 7-hour timer applied.


In [3]:
import torch
import os

INPUT_SEMANTIC = "/kaggle/input/datasets/vanditagupta2609/lmdb-input-2/lmdb-input/semantic.pt"
FIXED_SEMANTIC = "/kaggle/working/semantic_fixed.pt"
DATA_PATH = "/kaggle/input/datasets/vanditagupta2609/lmdb-input-2/lmdb-input/ffhq256.lmdb"
CHECKPOINT_DIR = '/kaggle/working/Patch-DM/checkpoints/patchdm_novelty/'

if os.path.exists(INPUT_SEMANTIC):
    data = torch.load(INPUT_SEMANTIC, map_location='cpu')
    if isinstance(data, dict) and 'model_state_dict' not in data:
        key = list(data.keys())[0]
        wrapped = {'model_state_dict': {'semantic_enc.weight': data[key]}}
    elif torch.is_tensor(data):
        wrapped = {'model_state_dict': {'semantic_enc.weight': data}}
    else:
        wrapped = data
    torch.save(wrapped, FIXED_SEMANTIC)
    print(f"✅ Semantic file ready at: {FIXED_SEMANTIC}")

✅ Semantic file ready at: /kaggle/working/semantic_fixed.pt


In [4]:
import subprocess
import time
import os

train_cmd = [
    "python", "-u", "train.py",
    "--batch_size", "4",
    "--patch_size", "64",
    "--data_path", DATA_PATH,
    "--name", "patchdm_novelty",
    "--semantic_path", FIXED_SEMANTIC
]

print("🔥 Launching Novelty Training (Will auto-stop after 7 hours)...")
process = subprocess.Popen(train_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

last_cleanup = time.time()

try:
    for line in process.stdout:
        print(line, end="")
        if time.time() - last_cleanup > 600:
            if os.path.exists(CHECKPOINT_DIR):
                files = [os.path.join(CHECKPOINT_DIR, f) for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.ckpt')]
                if len(files) > 2:
                    files.sort(key=os.path.getmtime)
                    for f in files[:-2]: 
                        os.remove(f)
                        print(f"\n🛡️ Disk Guardian: Removed {os.path.basename(f)}")
            last_cleanup = time.time()
except KeyboardInterrupt:
    process.terminate()

🔥 Launching Novelty Training (Will auto-stop after 7 hours)...
conf: patchdm_novelty
Global seed set to 0
Model params: 102.28 M
==Model size is 64==
ModelCheckpoint(save_last=True, save_top_k=-1, monitor=None) will duplicate the last checkpoint saved.
ckpt path: checkpoints/patchdm_novelty/last.ckpt
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/accelerator_connector.py:478: LightningDeprecationWarning: Setting `Trainer(gpus=[0])` is deprecated in v1.7 and will be removed in v2.0. Please use `Trainer(accelerator='gpu', devices=[0])` instead.
  rank_zero_deprecation(
Using 16bit None Automatic Mixed Precision (AMP)
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/plugins/precision/native_amp.py:47: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU ava

In [5]:
%%writefile calc_fid.py
import os
import torch
from torchvision.utils import save_image
from utils.dataset import FFHQlmdb
from templates import train_autoenc
from experiment import LitModel

print("1️⃣ Extracting 100 Real Images...")
os.makedirs('/kaggle/working/real_images', exist_ok=True)
dataset = FFHQlmdb(path="/kaggle/input/datasets/vanditagupta2609/lmdb-input-2/lmdb-input/ffhq256.lmdb", image_size=256)
for i in range(100):
    img_tensor = dataset[i]['img']
    img_tensor = (img_tensor + 1) / 2  # Denormalize from [-1, 1] to [0, 1]
    save_image(img_tensor, f'/kaggle/working/real_images/{i}.png')

print("2️⃣ Generating 100 Fake Images from Novelty Model...")
os.makedirs('/kaggle/working/fake_images', exist_ok=True)

# Find the latest checkpoint
ckpt_dir = '/kaggle/working/Patch-DM/checkpoints/patchdm_novelty/'
ckpts = [f for f in os.listdir(ckpt_dir) if f.endswith('.ckpt')]
ckpts.sort(key=lambda x: os.path.getmtime(os.path.join(ckpt_dir, x)))
latest_ckpt = os.path.join(ckpt_dir, ckpts[-1])
print(f"Loading checkpoint: {latest_ckpt}")

# Setup config and load model
conf = train_autoenc()
conf.batch_size = 4
conf.patch_size = 64
model = LitModel.load_from_checkpoint(latest_ckpt, conf=conf).cuda()
model.eval()

with torch.no_grad():
    for i in range(25):  # 25 batches * 4 images = 100 images
        samples = model.sample(N=4, device='cuda')
        for j in range(4):
            save_image(samples[j], f'/kaggle/working/fake_images/{(i*4)+j}.png')

print("✅ Image generation complete.")

Writing calc_fid.py


In [6]:
# Run the extraction and generation
!python calc_fid.py

# Calculate FID between the real 100 images and the fake 100 images
print("\n📊 Calculating FID Score...")
!python -m pytorch_fid /kaggle/working/real_images /kaggle/working/fake_images

# Package the weights and images
print("\n📦 Packaging final submission files...")
%cd /kaggle/working/
!zip -r /kaggle/working/patchdm_b23cm1061_novelty.zip /kaggle/working/Patch-DM/checkpoints/patchdm_novelty/ /kaggle/working/fake_images/

print("✅ DONE. Download 'patchdm_b23cm1061_novelty.zip' and record your FID score for the report.")

1️⃣ Extracting 100 Real Images...
2️⃣ Generating 100 Fake Images from Novelty Model...
Loading checkpoint: /kaggle/working/Patch-DM/checkpoints/patchdm_novelty/last.ckpt
Global seed set to 0
Model params: 68.10 M
==Model size is 64==
Traceback (most recent call last):
  File "/kaggle/working/Patch-DM/calc_fid.py", line 30, in <module>
    model = LitModel.load_from_checkpoint(latest_ckpt, conf=conf).cuda()
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py", line 139, in load_from_checkpoint
    return _load_from_checkpoint(  # type: ignore[return-value]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py", line 188, in _load_from_checkpoint
    return _load_state(cls, checkpoint, strict=strict, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.